# Fine-Tuning BERT for Railroad Condition Classification

Fine-tunes `bert-base-uncased` and `roberta-base` to classify DAS signal descriptions
into 4 condition classes from the CNN-LSTM-SW paper (Elsevier GEITS 2024).

**Task:** Text → {NC, TP, AC1, AC2}  
**GPU:** T4 (free Colab) — training takes ~15 minutes  
**Author:** Md Arifur Rahman | github.com/arifme071

**Paper:** Rahman MA et al. DOI: 10.1016/j.geits.2024.100178

In [ ]:
# Install dependencies
!pip install transformers datasets evaluate scikit-learn accelerate -q

In [ ]:
import json
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import evaluate
import torch

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Build training dataset ────────────────────────────────────────────────────
# Synthetic text descriptions of DAS signal conditions
# Generated from feature tables in Rahman et al. 2024

LABEL2ID = {"NC": 0, "TP": 1, "AC1": 2, "AC2": 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

SAMPLES = [
    # NC — Normal Condition
    ("Low amplitude uniform fiber-optic signal with stable spectral centroid and minimal kurtosis variation along 4km track section", "NC"),
    ("Background acoustic noise with consistent mean values across all 4160 channels, spectral centroid near 0.33Hz", "NC"),
    ("Regular low-level vibration detected on fiber-optic cable with no significant amplitude spikes or frequency anomalies", "NC"),
    ("Stable DAS signal with band energy ratio consistent with ambient environmental conditions, no anomalous patterns", "NC"),
    ("Normal condition signal: spectral contrast within expected range, kurtosis values near baseline, no train presence", "NC"),
    ("Uniform signal amplitude across all channels, time-frequency spectrogram shows no concentrated energy regions", "NC"),
    ("Low amplitude noise floor with spectral kurtosis near zero indicating Gaussian noise distribution", "NC"),
    ("Background rail track signal: steady mean values, minimal variance between adjacent channels", "NC"),
    ("Quiet section of track with no detected acoustic events, all features within 2 standard deviations of baseline", "NC"),
    ("Environmental noise only: wind and ambient vibration detected, no mechanical anomalies or train events", "NC"),
    ("Stable fiber-optic DAS reading with consistent spectral profile across the 4.16km measurement range", "NC"),
    ("Normal condition confirmed: low mean amplitude, symmetric distribution, no outliers in feature space", "NC"),
    # TP — Train Position
    ("Strong amplitude signal detected at 1,847m with sustained elevated mean value indicating active train presence", "TP"),
    ("Moving acoustic source detected along track with progressive channel activation pattern — train passage confirmed", "TP"),
    ("Significant spectral centroid drop from 0.33 to 0.08 between 1500m and 2000m indicating train position", "TP"),
    ("High amplitude burst propagating across fiber channels at consistent velocity matching locomotive speed", "TP"),
    ("Train position signal: elevated Pearson skewness, high kurtosis peak, sustained over 300+ consecutive channels", "TP"),
    ("Acoustic wave front detected moving at rail speed — spectral centroid suppressed indicating heavy wheel-rail contact", "TP"),
    ("Strong low-frequency energy burst in STFT spectrogram between 200Hz and 800Hz matching train wheel signature", "TP"),
    ("Progressive amplitude increase followed by gradual decrease consistent with train approaching and departing sensor zone", "TP"),
    ("Band energy ratio significantly elevated in low-frequency bands — characteristic pattern of locomotive passing", "TP"),
    ("Mean value spike of 287 units detected at 2,340m — 15x above baseline consistent with train position", "TP"),
    ("Multi-channel correlated high amplitude event lasting 45 seconds — speed and duration consistent with freight train", "TP"),
    # AC1 — Anomaly Class 1 (light)
    ("Localized amplitude irregularity at 3,124m with elevated spectral kurtosis suggesting minor wheel flat or surface defect", "AC1"),
    ("Periodic impact signature detected with 0.5 second interval — consistent with slight wheel out-of-round condition", "AC1"),
    ("Short-duration high-frequency spike at 2,891m with band energy concentrated above 2kHz — light rail surface anomaly", "AC1"),
    ("Minor spectral anomaly with kurtosis value 3.2x baseline but localized to 3 consecutive channels — AC1 class", "AC1"),
    ("Impulsive acoustic event with rapid rise and decay time — signature consistent with minor track surface irregularity", "AC1"),
    ("Low-severity anomaly: elevated spectral contrast at isolated location, no sustained disruption to adjacent channels", "AC1"),
    ("Single channel amplitude outlier at 1,756m with high kurtosis — potential wheel flat of less than 10mm depth", "AC1"),
    ("Light acoustic emission detected at rail joint area with frequency content above 3kHz — classified as minor anomaly", "AC1"),
    # AC2 — Anomaly Class 2 (heavy)
    ("Severe amplitude disruption at 2,847m with spectral centroid collapse and extremely elevated kurtosis — rail joint anomaly", "AC2"),
    ("Heavy rail defect signature: multiple consecutive channels affected, sustained high-energy emission in all frequency bands", "AC2"),
    ("Deep rail joint detected at 3,456m — spectral contrast 8x baseline, band energy ratio inverted indicating severe discontinuity", "AC2"),
    ("Major structural anomaly: kurtosis value 12.4 (5x AC1 threshold), affecting 15+ channels — requires immediate inspection", "AC2"),
    ("High severity acoustic emission at 1,203m — STFT shows broadband energy surge consistent with rail crack or heavy joint defect", "AC2"),
    ("AC2 classified event: persistent multi-channel disruption with inversion of normal spectral profile at 2,678m", "AC2"),
    ("Severe track anomaly: spectral kurtosis 18.7, band energy in high-frequency bands 12x normal — critical rail defect", "AC2"),
    ("Rail joint defect confirmed: amplitude spike magnitude 45,000 units — 30x NC baseline, sustained over 8 consecutive channels", "AC2"),
]

# Expand with variations to reach ~800 samples
import random
random.seed(42)
PREFIXES = ["DAS measurement:", "Fiber-optic reading:", "Signal analysis shows:",
            "Sensor data indicates:", "HTL loop recording:", ""]

expanded = []
for text, label in SAMPLES:
    expanded.append({"text": text, "label": LABEL2ID[label]})
    # Generate variants with prefix
    for _ in range(8):
        prefix = random.choice(PREFIXES)
        variant = f"{prefix} {text}" if prefix else text
        expanded.append({"text": variant.strip(), "label": LABEL2ID[label]})

random.shuffle(expanded)
df = pd.DataFrame(expanded)

print(f"Total samples: {len(df)}")
print(df['label'].map(ID2LABEL).value_counts())

In [ ]:
# ── Tokenize & build HuggingFace Dataset ─────────────────────────────────────
MODEL_NAME = "bert-base-uncased"  # swap to "roberta-base" for RoBERTa

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "validation": Dataset.from_pandas(val_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True)),
})

tokenized = dataset.map(tokenize, batched=True)
print(f"Train: {len(tokenized['train'])} | Val: {len(tokenized['validation'])} | Test: {len(tokenized['test'])}")

In [ ]:
# ── Build model ───────────────────────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params/1e6:.1f}M | Trainable: {trainable_params/1e6:.1f}M")

In [ ]:
# ── Training ──────────────────────────────────────────────────────────────────
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=preds, references=labels)

training_args = TrainingArguments(
    output_dir="./railroad-bert-checkpoint",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=50,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
# ── Evaluate on test set ──────────────────────────────────────────────────────
predictions = trainer.predict(tokenized["test"])
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

print("\n── Classification Report ──")
print(classification_report(y_true, y_pred,
      target_names=[ID2LABEL[i] for i in range(4)]))

print("\n── Confusion Matrix ──")
print(confusion_matrix(y_true, y_pred))

In [ ]:
# ── Push to HuggingFace Hub ───────────────────────────────────────────────────
# Set your HF token: from huggingface_hub import login; login()

HF_REPO = "arifme071/railroad-engineering-bert"

trainer.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
print(f"Model published: https://huggingface.co/{HF_REPO}")

In [ ]:
# ── Inference demo ────────────────────────────────────────────────────────────
from transformers import pipeline

clf = pipeline("text-classification",
               model=f"./{training_args.output_dir}",
               tokenizer=tokenizer)

test_inputs = [
    "Sharp amplitude spike at 2,847m with spectral centroid drop — rail joint anomaly",
    "Moving acoustic source detected along track — train passage at 1,500m",
    "Low background noise, all channels within normal range",
    "Localized short-duration spike at 3,124m — minor wheel flat suspected",
]

for text in test_inputs:
    result = clf(text)[0]
    print(f"Input: {text[:60]}...")
    print(f"Prediction: {result['label']} (confidence: {result['score']:.3f})\n")